# Two-stage Transfer Learning Task
In assignment 7A, a ChemBERTa model was fine-tuned to the ESOL dataset. Since that task took some considerable training time, the model was saved for further reuse, e.g. where only the regression head is retrained (which in contrast is a much cheaper operation).

Conceptually, this is a two-stage TL approach:

Foundation model ChemBERTa (general chemistry language) -> ESOL-tuned ChemBERTa (biased towards physicochemical descriptors) -> quick predictors (linear probes)

Reusing fine-tuned checkpoints as new model backbones is a routine operation to save computational time.

### Tasks
Note: The same random-state for splitting the dataset was used for all involved notebooks (`foundation_models.ipynb`, `7A_FineTuning.ipynb`).

1) Load the ESOL-tuned ChemBERTa model (encoder plus small regressor NN) and evaluate the predictions for the ESOL data (snippet provided)
2) In analogy to the notebook `foundation_models.ipynb` (session15/16), use the ESOL-tuned ChemBERTa model as fixed encoder and build a small machine learning model of your choice on top (e.g. ridge regression, RF, GB, ...)
3) Replace the dataset by the toxicity dataset (``tdc_ld50_zhu.csv``) and rerun the evaluation for the different transfer learning combinations (ChemBERTa+Regressor(retrain), ESOL-tuned ChemBERTa+Regressor(do not retrain), ESOL-tuned ChemBERTa+MLModel(retrain)), i.e. simply rerun the notebook with another dataset. Hint: you can crop the dataset size a bit by sampling so that retraining doesn't take too long (e.g. a GB model from task 2 took about 6 mins on my PC).
4) Complete the discussion points.


### Task 0: Import dependencies and data

In [1]:
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Load the data including train test-split (use the same as in the other examples!)

In [2]:
df = pd.read_csv("esol.csv")
# df = pd.read_csv("tdc_ld50_zhu.csv")

# drop rows just in case either smiles or logS values are missing. 
# It is crucial to have complete and labelled data for our exercise!
df.dropna(axis=0, inplace=True)

print(f"Dataset size: {len(df)}")

Dataset size: 1128


In [3]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

Reuse Dataset class from assignment 7A.

In [4]:
class ESOLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles = self.df.iloc[idx]["smiles"]
        label = self.df.iloc[idx]["logS"]
        # label = self.df.iloc[idx]["ld_50"]

        enc = self.tokenizer(
            smiles,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float)
        }


### Task 1:
Load and evaluate the ESOL-tuned ChemBERTa model.

In [5]:
# recreate model class
class chemberta_esol_regressor(nn.Module):
    
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.fc1 = nn.Linear(encoder.config.hidden_size, 256)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0]
        x = self.act(self.fc1(cls))
        return self.fc2(x).squeeze(-1)

In [6]:
# load the pretrained encoder
encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
tokenizer = AutoTokenizer.from_pretrained("chemberta_esol_encoder")

model = chemberta_esol_regressor(encoder)

# Load the pretrained weights for the regressor head:
head_state = torch.load("chemberta_esol_regressor_head.pt", map_location="cpu")

model.fc1.load_state_dict(head_state["fc1"])
model.fc2.load_state_dict(head_state["fc2"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<All keys matched successfully>

Initialise the dataset and the loader.

In [7]:

test_dataset = ESOLDataset(val_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64)


Evaluate the pretrained model:

In [8]:
# important: put model into evaluation mode (diables dropout and gradient)
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        y_true.append(batch["labels"].numpy())
        y_pred.append(preds.numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE:  {mae:.3f}")
print(f"Test R²:   {r2:.3f}")

Test RMSE: 1.262
Test MAE:  0.890
Test R²:   0.663


### Task 2:
Use the ESOL-tuned ChemBERTa model as fixed encoder only and use its output as training input for a small ML model (not a NN) of your choice (= new trainable head). 

You define the encoder and tokenizer in analogy to the `foundation-models.ipynb`, likewise the smiles_encoding function, but you may have to change some small details.

Hint: Since the regression model is not a NN, you could use `return np.vstack(all_embeddings)` so that the embeddings are nicely compatible with any scikit-learn models.

In [11]:
# Step 1: Create a function to get embeddings from SMILES
def get_embeddings(smiles_list, encoder, tokenizer, batch_size=32):
    """Convert SMILES strings to embeddings using the encoder."""
    all_embeddings = []
    
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        
        # Tokenize the batch
        enc = tokenizer(batch, truncation=True, padding=True, 
                        max_length=128, return_tensors="pt")
        
        # Get embeddings (no gradients needed)
        with torch.no_grad():
            outputs = encoder(enc["input_ids"], attention_mask=enc["attention_mask"])
            # Take the [CLS] token embedding (first token)
            cls_embeddings = outputs.last_hidden_state[:, 0].numpy()
            all_embeddings.append(cls_embeddings)
    
    return np.vstack(all_embeddings)

# Step 2: Get embeddings for training and validation
X_train = get_embeddings(train_df["smiles"].tolist(), encoder, tokenizer)
y_train = train_df["logS"].values

X_val = get_embeddings(val_df["smiles"].tolist(), encoder, tokenizer)
y_val = val_df["logS"].values

# Step 3: Train a classical ML model (e.g., Random Forest)
ml_model = RandomForestRegressor(n_estimators=100, random_state=42)
ml_model.fit(X_train, y_train)

# Step 4: Evaluate
y_pred = ml_model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val,y_pred)
r2 = r2_score(y_val, y_pred)

print(f"Random Forest")
print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE:  {mae:.3f}")
print(f"Test R²:   {r2:.3f}")

Random Forest
Test RMSE: 1.196
Test MAE:  0.866
Test R²:   0.697


### Task 3: Rerun with the toxicity dataset 
You can simply replace the imported dataframe. Note that depending on the model you chose in Task 2, training may take a bit - you can alleviate that problem by using a sample of the dataset.

You can run the General ChemBERTa + Regressor model in the original ``foundation_model.ipynb`` notebook (session 15/16).

In [12]:
# df = pd.read_csv("esol.csv")
df = pd.read_csv("tdc_ld50_zhu.csv")

# drop rows just in case either smiles or logS values are missing. 
# It is crucial to have complete and labelled data for our exercise!
df.dropna(axis=0, inplace=True)
print(f"Dataset size: {len(df)}")

Dataset size: 7376


In [13]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

Reuse Dataset class from assignment 7A.

In [15]:
class ESOLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles = self.df.iloc[idx]["smiles"]
        # label = self.df.iloc[idx]["logS"]
        label = self.df.iloc[idx]["ld_50"]

        enc = self.tokenizer(
            smiles,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float)
        }


In [16]:
# load the pretrained encoder
encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
tokenizer = AutoTokenizer.from_pretrained("chemberta_esol_encoder")

model = chemberta_esol_regressor(encoder)

# Load the pretrained weights for the regressor head:
head_state = torch.load("chemberta_esol_regressor_head.pt", map_location="cpu")

model.fc1.load_state_dict(head_state["fc1"])
model.fc2.load_state_dict(head_state["fc2"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<All keys matched successfully>

In [19]:
# Step 1: Create a function to get embeddings from SMILES
def get_embeddings(smiles_list, encoder, tokenizer, batch_size=32):
    """Convert SMILES strings to embeddings using the encoder."""
    all_embeddings = []
    
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        
        # Tokenize the batch
        enc = tokenizer(batch, truncation=True, padding=True, 
                        max_length=128, return_tensors="pt")
        
        # Get embeddings (no gradients needed)
        with torch.no_grad():
            outputs = encoder(enc["input_ids"], attention_mask=enc["attention_mask"])
            # Take the [CLS] token embedding (first token)
            cls_embeddings = outputs.last_hidden_state[:, 0].numpy()
            all_embeddings.append(cls_embeddings)
    
    return np.vstack(all_embeddings)

# Step 2: Get embeddings for training and validation
X_train = get_embeddings(train_df["smiles"].tolist(), encoder, tokenizer)
y_train = train_df["ld_50"].values

X_val = get_embeddings(val_df["smiles"].tolist(), encoder, tokenizer)
y_val = val_df["ld_50"].values

# Step 3: Train a classical ML model (e.g., Random Forest)
ml_model = RandomForestRegressor(n_estimators=100, random_state=42)
ml_model.fit(X_train, y_train)

# Step 4: Evaluate
y_pred = ml_model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val,y_pred)
r2 = r2_score(y_val, y_pred)

print(f"Random Forest with toxicity dataset")
print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE:  {mae:.3f}")
print(f"Test R²:   {r2:.3f}")

Random Forest with toxicity dataset
Test RMSE: 0.752
Test MAE:  0.570
Test R²:   0.413


In [20]:
import pandas as pd
from IPython.display import display

# Your results from the notebook
data = {
    "Model": [
        "ESOL-tuned + NN Head (Task 1)",
        "ESOL-tuned + Random Forest (Task 2)",
        "Toxicity + Random Forest (Task 3)"
    ],
    "Dataset": ["ESOL", "ESOL", "Toxicity"],
    "RMSE": [1.262, 1.196, 0.752],
    "R²": [0.663, 0.697, 0.413]
}

df_results = pd.DataFrame(data)

# Function to highlight best values in green
def highlight_best(val, col_name, best_rmse, best_r2):
    if col_name == "RMSE" and val == best_rmse:
        return 'background-color: lightgreen'
    elif col_name == "R²" and val == best_r2:
        return 'background-color: lightgreen'
    return ''

best_rmse = df_results["RMSE"].min()  # Lower RMSE is better
best_r2 = df_results["R²"].max()      # Higher R² is better

# Apply styling
styled_df = df_results.style.apply(
    lambda row: [highlight_best(row["RMSE"], "RMSE", best_rmse, best_r2) if col == "RMSE" 
                 else highlight_best(row["R²"], "R²", best_rmse, best_r2) if col == "R²" 
                 else '' for col in df_results.columns],
    axis=1
)

display(styled_df)

,Model,Dataset,RMSE,R²
0,ESOL-tuned + NN Head (Task 1),ESOL,1.262000,0.663000
1,ESOL-tuned + Random Forest (Task 2),ESOL,1.196000,0.697000
2,Toxicity + Random Forest (Task 3),Toxicity,0.752000,0.413000


### Task 4: Discussion

1) Why is it important for comparing the generalisation/performance of the different models to have the same random-state for the train-test split considering the fine-tuning in 7A and the evaluation in 7B? What would the predictions tell you otherwise?
- random state 42 ensures that all models have the same train and test data splits, ensures all models get them same data for fair comparison.
2) How did the performances of the three approaches compare for the ESOL dataset? Did the transfer learning stages improve the models?
- In comparison to the ESOL data set it was similar performance for both models, Random forest was slightly better which is known before running the models because of the small data set around 1100 entries which is not the best for NN.
- For both models Transfer learning was good, since both models predicted Solubility reasonably well check R^2 values 
3) Smaller models trained on molecular descriptors based on the smiles strings in the ESOL dataset (e.g. a GB model), delivered:
- Train RMSE: 0.386
- Test RMSE: 0.776
- Train R2: 0.965
- Test R2: 0.873
- How do you judge that in comparison to the much more complicated models?
- I would say on small data sets I would use a normal ML model, than using transformers since small ML models perform better than transfomers on a small data set ~1000 molecules, as seen here the R^2 values are super good in comparison to the one with transformers.
4) Discuss the results for using the three approaches on the toxicity dataset. Which one performed best? What is a clear no-go? Comment on target vs. source tasks in this context.
- The clear no-go is using the ESOL-tuned model directly on toxicity without retraining it was optimized for solubility (logS), not toxicity (LD₅₀). The source task (solubility) and target task (toxicity) are chemically different, so the model learned the wrong patterns
5) What would be a better approach for the toxicity?
- Fine-tune ChemBERTa directly on a toxicity dataset rather than routing through ESOL. Or use a model pre-trained on a task closer to toxicity.
6) How could we generally improve the performance?
- more training data (larger dataset)
- hyperparameter tunning
- better pretraining domain similar for toxicity for better prediction